# Load SN1a from file selected by Garazi

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-06-15
- **last update : 2026-06-24

Voilà le fichier CSV pour cette SN 


    FinkID = 313629129605382159, 
    RubinID = 739161397740437575). 

Dedans tu as :
- flux, flux_err : résultat de ma photométrie d'ouverture en électrons
- psfFlux, psfFluxErr : photométrie PSF de Rubin sur l'image de différence
- scienceFlux, scienceFluxErr : photométrie PSF de Rubin sur l'image de science
- mag, mag_err, flux_calib_njy, flux_calib_err_njy : résultat après avoir appliqué la fonction photoCalib de Rubin pour transformer le flux en électrons en flux en nanoJansky ou en magnitude

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import time
import warnings
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from astropy.time import Time
import astropy.table
from datetime import datetime, timedelta

import sncosmo

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

### 0.1 Utilities

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
# ── Photometric system ────────────────────────────────────────────────────────
RUBIN_ZP = 31.4  # AB zeropoint for nJy fluxes
ZPSYS = "ab"

# ── SALT model ────────────────────────────────────────────────────────────────
# Try salt3 first; fall back to salt2-extended if not available
SALT_SOURCES_PRIORITY = ["salt3", "salt2-extended"]
BAND_PREFIX = "lsst"  # sncosmo LSST band names: lsstu, lsstg, ...
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLOR = {"u": "b", "g": "green", "r": "red", "i": "orange", "z": "grey", "y": "k"}
# ── Quality cuts for SALT fit ─────────────────────────────────────────────────
SNR_MIN = 3.0  # minimum SNR per point (psfFlux / psfFluxErr)
MIN_DATAPOINTS = 5  # minimum points after quality cuts

# ── Two-pass free-z strategy ──────────────────────────────────────────────────
Z_BOUND_LO = 0.02
Z_BOUND_HI = 1.50
N_GRID_Z = 100
Z_REFINE_HALF = 0.15  # ±half-window around best grid z for free-z fit

# ── Tripp formula nuisance parameters ────────────────────────────────────────
ALPHA = 0.14  # Betoule+ 2014
BETA = 3.14  # Betoule+ 2014
M_B = -19.3  # absolute B-band magnitude

# ── Flat ΛCDM cosmology for Hubble diagram ────────────────────────────────────
H0 = 70.0
OmegaM = 0.30
OmegaL = 0.70

# ── Band colours ─────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#1f77b4",  # blue
    "g": "#2ca02c",  # green
    "r": "#d62728",  # red
    "i": "#ff7f0e",  # orange
    "z": "#8c564b",  # brown
    "y": "#9467bd",  # purple
}

In [ ]:
SALT_SOURCE = None
for src_name in SALT_SOURCES_PRIORITY:
    try:
        _model = sncosmo.Model(source=src_name)
        SALT_SOURCE = src_name
        print(f"SALT source selected: {SALT_SOURCE}")
        break
    except Exception as e:
        print(f"  {src_name} not available: {e}")

if SALT_SOURCE is None:
    raise RuntimeError("No SALT model available. Install sncosmo and its data: sncosmo.download_builtins()")

In [ ]:
def build_sncosmo_table(
    df: pd.DataFrame,
    mjd_col: str = "r:midpointMjdTai",
    flux_col: str = "r:psfFlux",
    fluxerr_col: str = "r:psfFluxErr",
    band_col: str = "r:band",
    snr_min: float = SNR_MIN,
    band_prefix: str = BAND_PREFIX,
    zp: float = RUBIN_ZP,
    zpsys: str = ZPSYS,
) -> astropy.table.Table:
    """Convert a Fink src DataFrame to an sncosmo-compatible observation Table.

    Returns an astropy Table with columns: time, band, flux, fluxerr, zp, zpsys.
    """
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    df = df.copy()
    df["snr_"] = df[flux_col] / df[fluxerr_col]

    # Quality cuts
    mask = df[fluxerr_col] > 0 & df[flux_col].notna() & df[fluxerr_col].notna() & (
        df["snr_"] >= snr_min
    ) & df[band_col].isin(BAND_ORDER)
    obs = df[mask].copy()

    print(f"Quality cuts: {len(df)} → {len(obs)} points (SNR ≥ {snr_min})")
    if len(obs) == 0:
        raise ValueError("No points survive quality cuts.")

    table = astropy.table.Table(
        {
            "time": obs[mjd_col].values.astype(float),
            "band": [f"{band_prefix}{b}" for b in obs[band_col].values],
            "flux": obs[flux_col].values.astype(float),
            "fluxerr": obs[fluxerr_col].values.astype(float),
            "zp": np.full(len(obs), zp),
            "zpsys": [zpsys] * len(obs),
        }
    )
    return table

In [ ]:
def make_sncosmo_table_fink(
    lc_df: pd.DataFrame,
    snr_min: float = SNR_MIN,
    zp: float = RUBIN_ZP,
    zpsys: str = ZPSYS,
    band_prefix: str = BAND_PREFIX,
) -> astropy.table.Table:
    """Convert a Fink LSST alert DataFrame to an astropy.Table for sncosmo.

    Input columns expected
    ----------------------
    r:midpointMjdTai  : observation epoch (MJD)
    r:band            : single-letter band name (u/g/r/i/z/y)
    r:psfFlux         : PSF flux in nJy
    r:psfFluxErr      : PSF flux uncertainty in nJy
    r:snr             : signal-to-noise ratio from Fink file (stored as snr_file)
    r:isDipole        : flag is there a dipole

    Returns
    -------
    astropy.Table with columns:
        time, band, flux, fluxerr, zp, zpsys, isDipole, snr_file

    The column `snr_file` carries the original `r:snr` value from the Fink alert
    file (after quality cuts), allowing plots to cross-check against the
    recomputed per-band SNR = sqrt(sum((flux/fluxerr)^2)).
    """
    df = lc_df.copy()

    # Rename to generic names
    df = df.rename(
        columns={
            "r:midpointMjdTai": "time",
            "r:band": "band_raw",
            "r:psfFlux": "flux",
            "r:psfFluxErr": "fluxerr",
            "r:snr": "snr",
            "r:isDipole": "isDipole",
        }
    )

    # Quality cuts
    df = df[df["fluxerr"] > 0].copy()
    if "snr" in df.columns:
        df = df[df["snr"] >= snr_min].copy()

    # Build sncosmo band names
    df["band"] = band_prefix + df["band_raw"].str.strip().str.lower()

    # snr_file: original r:snr from Fink file (NaN if column absent)
    snr_file_vals = df["snr"].values.astype(float) if "snr" in df.columns else np.full(len(df), np.nan)

    table = astropy.table.Table(
        {
            "time": df["time"].values.astype(float),
            "band": df["band"].values,
            "flux": df["flux"].values.astype(float),
            "fluxerr": df["fluxerr"].values.astype(float),
            "zp": np.full(len(df), zp, dtype=float),
            "zpsys": np.full(len(df), zpsys),
            "isDipole": df["isDipole"].values.astype(bool),
            "snr_file": snr_file_vals,
        }
    )
    return table


print("make_sncosmo_table_fink helper ready.")

In [ ]:
def fit_salt_fink(
    lc_df: pd.DataFrame,
    z_spec: float,
    fit_z: bool = False,
) -> dict:
    """Fit a Fink LSST SNIa light curve with SALT via sncosmo.

    Parameters
    ----------
    lc_df   : raw Fink alert DataFrame for one diaObjectId
    z_spec  : spectroscopic redshift (fixed during fit unless fit_z=True)
    fit_z   : if True, also free the redshift with a tight prior

    Returns
    -------
    dict with keys:
        success, result, fitted_model, table,
        z, t0, x0, x1, c,
        chi2, ndof, chi2_red,
        mB, mu_tripp, message
    """
    # Build the sncosmo observation table
    try:
        obs = make_sncosmo_table_fink(lc_df)
    except Exception as exc:
        return {"success": False, "message": f"Table build failed: {exc}"}

    if len(obs) < MIN_DATAPOINTS:
        return {
            "success": False,
            "message": f"Only {len(obs)} points after quality cuts (need {MIN_DATAPOINTS})",
        }

    # Initialise SALT2 model
    model = sncosmo.Model(source=SALT_SOURCE)

    # Initial parameter guesses
    peak_idx = int(np.argmax(obs["flux"]))
    t0_guess = float(obs["time"][peak_idx])
    # Rough x0 from peak flux: F_peak ≈ x0 * model_peak  → use 1e-3 as safe default
    model.set(z=z_spec, t0=t0_guess, x0=1e-3, x1=0.0, c=0.0)

    vparam_names = ["t0", "x0", "x1", "c"]
    bounds = {
        "t0": (t0_guess - 40.0, t0_guess + 40.0),
        "x0": (1e-10, 1e2),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ["z", "t0", "x0", "x1", "c"]
        dz = min(0.05 * z_spec, 0.1)
        bounds["z"] = (max(z_spec - dz, 1e-4), z_spec + dz)

    # Run fit
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result, fitted_model = sncosmo.fit_lc(
                obs,
                model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,
                warn=False,
            )
    except Exception as exc:
        return {"success": False, "message": f"Fit failed: {exc}", "table": obs}

    chi2 = result.chisq
    ndof = result.ndof
    chi2_red = chi2 / max(ndof, 1)

    # ── Tripp distance modulus ────────────────────────────────────────────────
    z_fit = float(fitted_model["z"])
    x0_fit = float(fitted_model["x0"])
    x1_fit = float(fitted_model["x1"])
    c_fit = float(fitted_model["c"])
    # m_B = -2.5 * log10(x0) + ZP_SALT2  (SALT2 internal ZP ≈ 10.635)
    # Standard formula: m_B^* = -2.5*log10(x0) + 10.635
    SALT2_ZP_INTERNAL = 10.635
    mB = -2.5 * np.log10(max(x0_fit, 1e-30)) + SALT2_ZP_INTERNAL
    mu = mB - M_B + ALPHA * x1_fit - BETA * c_fit

    return {
        "success": True,
        "result": result,
        "fitted_model": fitted_model,
        "table": obs,
        "z": z_fit,
        "t0": float(fitted_model["t0"]),
        "x0": x0_fit,
        "x1": x1_fit,
        "c": c_fit,
        "chi2": chi2,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "mB": mB,
        "mu_tripp": mu,
        "message": result.message,
    }


print("fit_salt2_fink helper ready.")

In [ ]:
def plot_salt_fit_fink(ax, oid: int, name: str, res: dict, z_spec: float):
    """
    Plot raw Fink data points + SALT model curves for one object.

    Legend shows per-band SNR(b) = sqrt(sum((f/sigma)^2)) next to each band
    label (ncol=1).  The legend title displays:
      - SNR_tot = sqrt(sum_b SNR(b)^2)  recomputed from flux/fluxerr
      - snr_file = sqrt(sum(r:snr^2))   from the Fink file column

    Parameters
    ----------
    ax      : matplotlib Axes
    oid     : diaObjectId
    name    : display name (TNS name + type)
    res     : result dict from fit_salt2_fink
    z_spec  : spectroscopic redshift
    """
    if not res.get("success"):
        ax.set_title(f"{name}\nFit failed: {res.get('message', '')}", fontsize=7, color="red")
        ax.text(
            0.5, 0.5, "FIT FAILED", transform=ax.transAxes, ha="center", va="center", color="red", fontsize=12
        )
        return

    obs = res["table"]
    fitted_model = res["fitted_model"]
    t0 = res["t0"]
    chi2_red = res["chi2_red"]

    # convert into readable date
    tim = Time(t0, format="mjd")
    date_iso = tim.to_value("iso", subfmt="date")

    # ── Compute total SNR from flux/fluxerr (recomputed) ──────────────────────
    flux_all = np.array(obs["flux"], dtype=float)
    ferr_all = np.array(obs["fluxerr"], dtype=float)
    snr_tot = float(np.sqrt(np.sum((flux_all / ferr_all) ** 2)))

    # ── Compute total SNR from the Fink file column r:snr ─────────────────────
    if "snr_file" in obs.colnames:
        snr_file_col = np.array(obs["snr_file"], dtype=float)
        valid_snr = np.isfinite(snr_file_col)
        snr_file_tot = float(np.sqrt(np.sum(snr_file_col[valid_snr] ** 2))) if valid_snr.any() else np.nan
    else:
        snr_file_tot = np.nan

    # Dense time grid for model evaluation
    t_min = float(obs["time"].min())
    t_max = float(obs["time"].max())
    t_dense = np.linspace(t_min - 15, t_max + 15, 500)

    # loop on bands
    for b in BAND_ORDER:
        bname = BAND_PREFIX + b
        color = BAND_COLOR.get(b, "gray")

        # Data points
        mask = np.array(obs["band"]) == bname
        mask_dipole = mask & np.array(obs["isDipole"].astype(bool))

        ax.scatter(
            obs["time"][mask_dipole] - t0,
            obs["flux"][mask_dipole],
            color="grey",
            marker="o",
            s=100,
            facecolor="none",
        )

        if mask.sum() > 0:
            # Per-band SNR = sqrt(sum((f_i / sigma_i)^2))
            f_b = np.array(obs["flux"][mask], dtype=float)
            fe_b = np.array(obs["fluxerr"][mask], dtype=float)
            snr_b = float(np.sqrt(np.sum((f_b / fe_b) ** 2)))

            ax.errorbar(
                obs["time"][mask] - t0,
                obs["flux"][mask],
                yerr=obs["fluxerr"][mask],
                color=color,
                fmt="o",
                markersize=4,
                capsize=2,
                label=f"{b}  SNR={snr_b:.1f}",
                zorder=3,
            )

        # SALT2 model curve
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=RUBIN_ZP, zpsys=ZPSYS)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(t_dense[valid] - t0, f_model[valid], color=color, lw=1.5, zorder=2)
        except Exception:
            pass

    ax.axhline(0.0, color="k", lw=0.5, ls="--")
    ax.set_title(
        f"{name}  z={z_spec:.4f}  tmax={date_iso}\nx1={res['x1']:+.2f}  c={res['c']:+.2f}  χ²/dof={chi2_red:.2f}",
        fontsize=12,
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=10)
    ax.set_ylabel("psfFlux [nJy]", fontsize=10)
    ax.tick_params(axis="both", labelsize=10)
    # Legend title shows total SNR (recomputed) and SNR from Fink file
    # legend_title = f"SNR_tot={snr_tot:.1f}  snr_file={snr_file_tot:.1f}"
    legend_title = f"SNR_tot={snr_tot:.1f}"
    ax.legend(fontsize=8, ncol=1, loc="upper right", title=legend_title, title_fontsize=8)

## 1 — Configuration

In [ ]:
# ── Target object ─────────────────────────────────────────────────────────────
DIA_OBJECT_ID = 739161397740437575  # r:diaObjectId (Rubin)
FINK_ID = 313629129605382159  # Fink internal objectId (used for forcedphotometry)

In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB97_01_SNIaGarazi"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f"diaObjectId  : {DIA_OBJECT_ID}")
print(f"Fink ID      : {FINK_ID}")
print(f"Data dir     : {DATA_DIR.resolve()}")
print(f"Figures dir  : {FIGS_DIR.resolve()}")

In [ ]:
# input files
filename_rubin = f"{DATA_DIR}/lightcurve_tract_2704_i_739161397740437575.csv"
filename_fink = f"{DATA_DIR}/313629129605382159.csv"

## 2 — Load data and check

In [ ]:
df_r = pd.read_csv(filename_rubin)
df_f = pd.read_csv(filename_fink)

In [ ]:
index_znota = ~df_f["f:xm_legacydr8_zphot"].isna()
redshift_legdr8 = df_f["f:xm_legacydr8_zphot"][index_znota].values[0]
redshifterr_legdr8 = df_f["f:xm_legacydr8_e_zphot"][index_znota].values[0]
print(f"redshift = {redshift_legdr8} +/- {redshifterr_legdr8}")

In [ ]:
fit_fixed_z = True
fit_fixed_z_cov = redshifterr_legdr8**2
z_used = redshift_legdr8
z_source = redshift_legdr8
fit_free_z = None

In [ ]:
list_of_bands = df_f["r:band"].unique()
list_bands_ordered = []
for band in BAND_ORDER:
    if band in list_of_bands:
        list_bands_ordered.append(band)

In [ ]:
def plot_lightcurves(ax, df, x_col="r:midpointMjdTai", y_col="r:psfFlux", yerr_col="r:psfFluxErr"):
    """ """

    flag_ax_provided = True
    if ax is None:
        flag_ax_provided = False
        ax.subplots(1, 1, figsize=(12, 6))

    for band in list_bands_ordered:
        df = df_f[df_f["r:band"] == band]
        ax.errorbar(
            df["r:midpointMjdTai"],
            df["r:psfFlux"],
            yerr=df["r:psfFluxErr"],
            fmt="o",  # <-- pas de "-"
            ms=4,
            lw=1.2,
            capsize=3,
            color=BAND_COLOR[band],
            label=f"{band}",
        )

    return ax

In [ ]:
list(df_f.columns)

In [ ]:
df_r.columns

In [ ]:
df_r["date"]

In [ ]:
df_r["date"] = pd.to_datetime(df_r["date"])

In [ ]:
tr = Time(df_r["date"].values)
df_r["mjd"] = tr.mjd
df_r = df_r.sort_values("mjd")

In [ ]:
def mjd_to_datestr(mjd_array) -> list:
    """
    Convert an array of MJD (TAI) values to ISO date strings 'YYYY-MM-DD'.

    Uses astropy.time.Time for the conversion.
    """
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def add_date_axis_on_top(ax, mjd_values: np.ndarray, n_ticks: int = 7) -> None:
    """
    Add a secondary x-axis on **top** of *ax* showing calendar dates (YYYY-MM-DD),
    inclined 40 degrees to the left for readability.
    """
    finite = mjd_values[np.isfinite(mjd_values)]
    if len(finite) < 2:
        return

    mjd_lo, mjd_hi = float(finite.min()), float(finite.max())
    if mjd_hi <= mjd_lo:
        return

    n_ticks = max(3, min(n_ticks, len(finite)))
    tick_mjd = np.linspace(mjd_lo, mjd_hi, n_ticks)
    tick_lbls = mjd_to_datestr(tick_mjd)

    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(tick_mjd)
    ax_top.set_xticklabels(tick_lbls, rotation=40, ha="left", fontsize=7)
    ax_top.tick_params(axis="x", length=4, pad=2)
    ax_top.set_xlabel("Date (UTC)", fontsize=12, labelpad=6)


print("mjd_to_datestr() and add_date_axis_on_top() defined.")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax = plot_lightcurves(ax, df_f)
add_date_axis_on_top(ax, df_f["r:midpointMjdTai"].values)
ax.legend(title="psfFlux( Fink)")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True, layout="constrained")

ax1 = plot_lightcurves(ax1, df_f)
ax1.grid()
ax1.legend(title="psfFlux( Fink)", loc="upper left")

ax2.errorbar(
    df_r.mjd,
    df_r.flux,
    yerr=df_r.flux_err,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="blue",
    label="aperture flux (Garazi)",
)


ax2.errorbar(
    df_r.mjd,
    df_r.scienceFlux,
    yerr=df_r.scienceFluxErr,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="purple",
    label="scienceFlux",
)


ax2.errorbar(
    df_r.mjd,
    df_r.psfFlux,
    yerr=df_r.psfFluxErr,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="red",
    label="psfFlux (Dia)",
)

add_date_axis_on_top(ax1, df_r.mjd)

ax2.grid()
ax2.set_xlabel("Date (MJD)", fontsize=12, labelpad=6)
ax2.legend(title="Flux in Rubin DP2", loc="upper left")
# plt.tight_layout()
plt.show()

## 3 — Baseline subtraction and ApertureFlux recalibration

### Strategy

**Baseline subtraction** (both Fink and Rubin curves):
We define the pre-explosion baseline as the median flux over all epochs *before*
the estimated explosion MJD (`MJD_EXPLOSION`).  
The baseline is then subtracted from every point of the corresponding curve, so
that the light curve rises from zero.

**ApertureFlux recalibration** (Rubin subplot only):
Garazi's aperture photometry (`flux`, `flux_err`) is in electrons, not in nJy.
We estimate the conversion factor as:
$$\text{scale} = \mathrm{median}\!\left(\frac{\texttt{psfFlux}_{\text{post}}}{\texttt{flux}_{\text{post}}}\right)$$
computed over post-explosion epochs with good SNR for both quantities.
The scaled aperture flux `flux_calib = scale × flux` (and its error) is then
baseline-subtracted like the other Rubin curves.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 3 — Baseline subtraction & ApertureFlux recalibration
# ─────────────────────────────────────────────────────────────────────────────

# ── 3.0  Explosion epoch estimate ────────────────────────────────────────────
# Use the epoch of maximum psfFlux in the Fink i-band light curve as a rough
# proxy for peak.  The 'pre-explosion' window is everything before this epoch.
# You can override MJD_EXPLOSION manually if you have a better prior.

band_ref = "i"  # reference band used for explosion-epoch detection
df_fi = df_f[df_f["r:band"] == band_ref].copy()

if len(df_fi) == 0:
    # Fallback: use all Fink bands
    df_fi = df_f.copy()

idx_peak_fink = df_fi["r:psfFlux"].idxmax()
MJD_PEAK_FINK = float(df_fi.loc[idx_peak_fink, "r:midpointMjdTai"])

# Explosion is estimated as ~ 15 days before Fink peak (typical SNIa rise time).
# Adjust RISE_TIME_DAYS if needed.
RISE_TIME_DAYS = 15.0
MJD_EXPLOSION = MJD_PEAK_FINK - RISE_TIME_DAYS

print(f"Fink i-band peak MJD  : {MJD_PEAK_FINK:.3f}")
print(f"Estimated explosion   : {MJD_EXPLOSION:.3f}  (peak - {RISE_TIME_DAYS} d)")


# ── 3.1  Fink: per-band baseline subtraction ──────────────────────────────────
# For each band, compute the median psfFlux over pre-explosion epochs and
# subtract it from all points in that band.

df_f = df_f.copy()  # avoid SettingWithCopyWarning
df_f["psfFlux_bsub"] = np.nan
df_f["psfFluxErr_bsub"] = np.nan

baseline_fink = {}  # store per-band baseline values for reference

for band in df_f["r:band"].unique():
    mask_band = df_f["r:band"] == band
    mask_pre = mask_band & (df_f["r:midpointMjdTai"] < MJD_EXPLOSION)

    pre_flux = df_f.loc[mask_pre, "r:psfFlux"]
    if len(pre_flux) >= 2:
        baseline = float(np.median(pre_flux))
    elif len(pre_flux) == 1:
        baseline = float(pre_flux.iloc[0])
    else:
        # No pre-explosion point in this band: use the global minimum as proxy
        baseline = float(df_f.loc[mask_band, "r:psfFlux"].min())
        print(f"  [{band}] No pre-explosion point found — using flux min as baseline")

    # I don't want to subtract any baselne
    baseline = 0

    baseline_fink[band] = baseline
    df_f.loc[mask_band, "psfFlux_bsub"] = df_f.loc[mask_band, "r:psfFlux"] - baseline
    df_f.loc[mask_band, "psfFluxErr_bsub"] = df_f.loc[mask_band, "r:psfFluxErr"]
    print(f"  Fink band '{band}': baseline = {baseline:.3f} nJy  ({len(pre_flux)} pre-SN points)")

In [ ]:
# ── 3.2  Rubin: baseline subtraction for psfFlux and scienceFlux ─────────────
# For this dataset only band i is present in df_r; the logic is band-agnostic
# so it will generalise when more bands become available.

df_r = df_r.copy()
mask_r_pre = df_r["mjd"] < MJD_EXPLOSION

# psfFlux baseline
pre_psf = df_r.loc[mask_r_pre, "psfFlux"]
# baseline_r_psf = float(np.median(pre_psf)) if len(pre_psf) >= 1 else 0.0
baseline_r_psf = float(np.nanmedian(pre_psf)) if len(pre_psf) >= 1 else 0.0

df_r["psfFlux_bsub"] = df_r["psfFlux"] - baseline_r_psf
df_r["psfFluxErr_bsub"] = df_r["psfFluxErr"]
print(f"\nRubin psfFlux  baseline = {baseline_r_psf:.3f} nJy  ({len(pre_psf)} pre-SN points)")

# scienceFlux baseline
pre_sci = df_r.loc[mask_r_pre, "scienceFlux"]
# baseline_r_sci = float(np.median(pre_sci)) if len(pre_sci) >= 1 else 0.0
baseline_r_sci = float(np.nanmedian(pre_sci)) if len(pre_sci) >= 1 else 0.0
df_r["scienceFlux_bsub"] = df_r["scienceFlux"] - baseline_r_sci
df_r["scienceFluxErr_bsub"] = df_r["scienceFluxErr"]
print(f"Rubin scienceFlux baseline = {baseline_r_sci:.3f} nJy  ({len(pre_sci)} pre-SN points)")


# aperture baseline
pre_aper = df_r.loc[mask_r_pre, "flux"]
baseline_r_aper = float(np.nanmedian(pre_aper)) if len(pre_aper) >= 1 else 0.0
df_r["flux_bsub"] = df_r["flux"] - baseline_r_aper
df_r["flux_err_bsub"] = df_r["flux_err"]
print(f"Aperture flux  baseline = {baseline_r_aper:.3f} units  ({len(pre_aper)} pre-SN points)")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6), layout="constrained")

ax.errorbar(
    df_r.mjd,
    df_r["flux_bsub"],
    yerr=df_r["flux_err_bsub"],
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="blue",
    label="aperture flux (Garazi)",
)


ax.errorbar(
    df_r.mjd,
    df_r.scienceFlux_bsub,
    yerr=df_r.scienceFluxErr_bsub,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="purple",
    label="scienceFlux",
)


ax.errorbar(
    df_r.mjd,
    df_r.psfFlux_bsub,
    yerr=df_r.psfFluxErr_bsub,
    fmt="o",  # <-- pas de "-"
    ms=4,
    lw=1.2,
    capsize=3,
    color="red",
    label="psfFlux (Dia)",
)

add_date_axis_on_top(ax, df_r.mjd)

ax.grid()
ax.set_xlabel("Date (MJD)", fontsize=12, labelpad=6)
ax.legend(title="Flux in Rubin DP2", loc="upper left")
# plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3  ApertureFlux recalibration (electrons → nJy) ─────────────────────────
# The scale factor is derived from the ratio psfFlux / aperture_flux on
# post-explosion epochs where both measurements have good SNR.

SNR_CALIB_MIN = 3.0  # minimum SNR required for calibration points

mask_r_post = df_r["mjd"] >= MJD_EXPLOSION
snr_psf = df_r["psfFlux"].abs() / df_r["psfFluxErr"].replace(0, np.nan)
snr_aper = df_r["flux"].abs() / df_r["flux_err"].replace(0, np.nan)

print(snr_psf)
print(snr_aper)

mask_calib = (
    mask_r_post
    #    & (snr_psf  >= SNR_CALIB_MIN)
    #    & (snr_aper >= SNR_CALIB_MIN)
    & (df_r["flux"].abs() > 0)
)

ratios = df_r.loc[mask_calib, "psfFlux"] / df_r.loc[mask_calib, "flux"]

if len(ratios) >= 3:
    # Robust estimate: median after 3-sigma clipping
    med = float(np.median(ratios))
    sigma = float(np.std(ratios))
    keep = ratios[(ratios - med).abs() < 3 * sigma]
    APERTURE_SCALE = float(np.median(keep))
    print(
        f"\nApertureFlux→nJy scale factor: {APERTURE_SCALE:.4f}  "
        f"(from {len(keep)} calibration points after 3σ clip)"
    )
elif len(ratios) >= 1:
    APERTURE_SCALE = float(np.median(ratios))
    print(
        f"\nApertureFlux→nJy scale factor: {APERTURE_SCALE:.4f}  "
        f"(from {len(ratios)} points — too few for sigma-clip)"
    )
else:
    APERTURE_SCALE = 1.0
    print("WARNING: No calibration points found — APERTURE_SCALE set to 1.0")

# Apply scale and baseline-subtract
df_r["flux_calib_nJy"] = df_r["flux"] * APERTURE_SCALE
df_r["flux_err_calib_nJy"] = df_r["flux_err"] * APERTURE_SCALE

pre_aper_calib = df_r.loc[mask_r_pre, "flux_calib_nJy"]
baseline_r_aper = float(np.median(pre_aper_calib)) if len(pre_aper_calib) >= 1 else 0.0
df_r["flux_calib_bsub"] = df_r["flux_calib_nJy"] - baseline_r_aper
df_r["flux_err_calib_bsub"] = df_r["flux_err_calib_nJy"]
print(f"Aperture flux (calib) baseline = {baseline_r_aper:.3f} nJy")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Section 3 — Comparison figure: Fink psfFlux (top) vs Rubin recalibrated
#             fluxes (bottom), all baseline-subtracted
# ─────────────────────────────────────────────────────────────────────────────

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
fig.subplots_adjust(hspace=0.08)

# ── Subplot 1 : Fink psfFlux (baseline-subtracted) ───────────────────────────
for band in list_bands_ordered:
    df_band = df_f[df_f["r:band"] == band]
    ax1.errorbar(
        df_band["r:midpointMjdTai"],
        df_band["psfFlux_bsub"],
        yerr=df_band["psfFluxErr_bsub"],
        fmt="o",
        ms=4,
        lw=1.2,
        capsize=3,
        color=BAND_COLOR[band],
        label=f"Fink psfFlux – {band}",
    )

# Mark the estimated explosion epoch
ax1.axvline(MJD_EXPLOSION, color="grey", ls="--", lw=1, alpha=0.7, label=f"MJD_explosion={MJD_EXPLOSION:.1f}")
ax1.axhline(0, color="grey", ls=":", lw=0.8)
ax1.set_ylabel("psfFlux (nJy)", fontsize=11)
ax1.legend(title="Fink (forced photometry)", loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.35)
ax1.set_title(
    f"SNIa — FinkID {FINK_ID}  /  diaObjectId {DIA_OBJECT_ID}\n"
    f"z = {z_used:.4f}  |  baseline subtracted  |  explosion MJD ≈ {MJD_EXPLOSION:.1f}",
    fontsize=11,
)

# Add date axis on top
add_date_axis_on_top(ax1, df_f["r:midpointMjdTai"].values)


# ── Subplot 2 : Rubin fluxes (baseline-subtracted & ApertureFlux recalibrated)

# psfFlux (Rubin DIA) — in nJy, baseline-subtracted
ax2.errorbar(
    df_r["mjd"],
    df_r["psfFlux_bsub"],
    yerr=df_r["psfFluxErr_bsub"],
    fmt="s",
    ms=5,
    lw=1.2,
    capsize=3,
    color="red",
    label="Rubin psfFlux (nJy, bsub)",
)

# scienceFlux (Rubin) — in nJy, baseline-subtracted
ax2.errorbar(
    df_r["mjd"],
    df_r["scienceFlux_bsub"],
    yerr=df_r["scienceFluxErr_bsub"],
    fmt="D",
    ms=4,
    lw=1.2,
    capsize=3,
    color="purple",
    label="Rubin scienceFlux (nJy, bsub)",
)

# ApertureFlux (Garazi) — recalibrated to nJy, baseline-subtracted
ax2.errorbar(
    df_r["mjd"],
    # df_r["flux_calib_bsub"],
    # yerr=df_r["flux_err_calib_bsub"],
    df_r["flux_bsub"],
    yerr=df_r["flux_err_bsub"],
    fmt="o",
    ms=4,
    lw=1.2,
    capsize=3,
    color="royalblue",
    # label=f"Aperture flux (Garazi × {APERTURE_SCALE:.3f}, nJy, bsub)",
    label=f"Aperture flux (Garazi  (nJy, bsub)",
)

ax2.axvline(MJD_EXPLOSION, color="grey", ls="--", lw=1, alpha=0.7)
ax2.axhline(0, color="grey", ls=":", lw=0.8)
ax2.set_ylabel("Baseline-sub flux (nJy)", fontsize=11)
ax2.set_xlabel("MJD (TAI)", fontsize=12)
ax2.legend(
    # title=f"Rubin DRP2 — band i  |  scale={APERTURE_SCALE:.3f} e⁻/nJy",
    title=f"Rubin DP2 — band i",
    loc="upper left",
    fontsize=9,
)
ax2.grid(True, alpha=0.35)

plt.tight_layout()
figname = FIGS_DIR / "lightcurves_bsub_recalib.png"
fig.savefig(figname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {figname}")

## 3 - Fitting

### 3.1 Generate the table

In [ ]:
obs_table = build_sncosmo_table(df_f)
print(f"sncosmo table: {len(obs_table)} points across bands {set(obs_table['band'])}")
obs_table

In [ ]:
fit_fixed_z = None
fit_fixed_z_cov = None

if z_used is not None and len(obs_table) >= MIN_DATAPOINTS:
    model = sncosmo.Model(source=SALT_SOURCE)
    model.set(z=z_used)

    # Estimate t0 from peak flux
    t0_init = float(obs_table["time"][np.argmax(obs_table["flux"])])
    model.set(t0=t0_init)

    print(f"Fitting {SALT_SOURCE} with z fixed = {z_used:.5f}  (source: {z_source})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            res, fitted_model = sncosmo.fit_lc(
                obs_table,
                model,
                ["t0", "x0", "x1", "c"],
                bounds={"x1": (-5, 5), "c": (-1, 2)},
            )
            fit_fixed_z = res
            fit_fixed_z_cov = res.covariance
            print(f"  status  : {res.message}")
            print(f"  t0      : {res.parameters[res.param_names.index('t0')]:.3f}")
            print(f"  x0      : {res.parameters[res.param_names.index('x0')]:.4e}")
            print(f"  x1      : {res.parameters[res.param_names.index('x1')]:.4f}")
            print(f"  c       : {res.parameters[res.param_names.index('c')]:.4f}")
            print(f"  χ²/dof  : {res.chisq:.2f} / {res.ndof}")
        except Exception as exc:
            print(f"  [error] fit failed: {exc}")
else:
    print("Skipping fixed-z fit (no redshift or insufficient data).")

In [ ]:
# ── Plot SALT3 fit (fixed z) ──────────────────────────────────────────────────
if fit_fixed_z is not None:
    fig = sncosmo.plot_lc(
        obs_table,
        model=fitted_model,
        errors=res.errors,
        ncol=4,
        # figtext=(
        #    f" diaObj(DP2)   + {str(DIA_OBJECT_ID)} \n"
        #    f" z={z_used:.2f} ({z_source:.2f})\n"
        #    f" x1={res.parameters[res.param_names.index('x1')]:.2f}  "
        #    f" c={res.parameters[res.param_names.index('c')]:.2f}  "
        #    f" χ²/dof={res.chisq:.1f}/{res.ndof}"
        # ),
        figtext=(""),
    )
    plt.tight_layout()
    fig.savefig(FIGS_DIR / "salt3_fit_fixed_z_subplots.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved → {FIGS_DIR}/salt3_fit_fixed_z_subplots.png")

## Another way for doing the fit.

- the goal is to a a single subplot with all light curves

In [ ]:
res = fit_salt_fink(lc_df=df_f, z_spec=z_used, fit_z=False)

In [ ]:
# plot_salt_fit_fink(ax, oid: int, name: str, res = res, z_spec= z_used)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# ── Subplot 1 : Fink psfFlux (baseline-subtracted) ───────────────────────────
plot_salt_fit_fink(ax, oid=FINK_ID, name=f"SNIa : {FINK_ID}", res=res, z_spec=z_used)
fig.savefig(FIGS_DIR / "salt3_fit_fixed_z_oneplot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {FIGS_DIR}/salt3_fit_fixed_z_oneplot.png")

## 4 — Ellipses de corrélation Fisher (SALT3)

On extrait la matrice de covariance retournée par `sncosmo.fit_lc` et on trace
les ellipses de corrélation à 68 % et 95 % de niveau de confiance (Δχ² = 2.30 et 6.17
pour 2 degrés de liberté).

**Paramètres affichés** : `t0`, `x0`, `x1`, `c` — les quatre paramètres libres du fit.
La sous-figure diagonale montre la gaussienne marginale 1D ; les sous-figures
hors-diagonale (triangle inférieur) montrent les ellipses 2D.

In [ ]:
# ndeg = len(res["result"].vparam_names)
ndeg = 2  # compare one parameter wrt another

In [ ]:
# ── Imports supplémentaires pour les ellipses Fisher ─────────────────────
from matplotlib.patches import Ellipse
from scipy.stats import chi2 as scipy_chi2

# Niveaux Δχ² pour 2 degrés de liberté
DELTA_CHI2_68 = scipy_chi2.ppf(0.683, df=ndeg)  # ≈ 2.30
DELTA_CHI2_95 = scipy_chi2.ppf(0.954, df=ndeg)  # ≈ 6.17

print(f"Δχ² 68% = {DELTA_CHI2_68:.4f}   Δχ² 95% = {DELTA_CHI2_95:.4f}")

In [ ]:
def plot_fisher_ellipses(
    param_names,
    theta_hat,
    cov,
    levels=(2.30, 6.17),
    figsize=None,
    title=None,
):
    """
    Trace les ellipses de corrélation Fisher à partir d'une matrice de covariance.

    Paramètres
    ----------
    param_names : list[str]          noms des paramètres (N)
    theta_hat   : array-like (N,)    valeurs centrales (best-fit)
    cov         : array-like (N, N)  matrice de covariance
    levels      : tuple de Δχ²       (68 %, 95 %) → (2.30, 6.17)
    figsize     : tuple ou None      taille de la figure (défaut : 2*N × 2*N)
    title       : str ou None        titre général de la figure

    Retourne
    --------
    fig : matplotlib.figure.Figure
    """
    n = len(param_names)
    if figsize is None:
        figsize = (2.5 * n, 2.5 * n)

    cov = np.asarray(cov)
    theta_hat = np.asarray(theta_hat)

    fig, axes = plt.subplots(n, n, figsize=figsize)
    if n == 1:
        axes = np.array([[axes]])

    colors = ["C0", "C1"]  # couleurs pour les deux niveaux
    labels = [r"68 % CL", r"95 % CL"]

    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            if i == j:
                # ── Diagonale : gaussienne marginale 1D ───────────────────
                sigma = np.sqrt(cov[i, i])
                x = np.linspace(theta_hat[i] - 4 * sigma, theta_hat[i] + 4 * sigma, 300)
                y = np.exp(-((x - theta_hat[i]) ** 2) / (2 * sigma**2))
                ax.plot(x, y, color="C0")
                ax.axvline(theta_hat[i], color="k", lw=0.8, ls="--")
                ax.fill_between(x, y, where=(np.abs(x - theta_hat[i]) <= sigma), alpha=0.25, color="C0")
                ax.set_yticks([])
                ax.set_ylim(0, 1.15)

            elif i > j:
                # ── Triangle inférieur : ellipses 2D ──────────────────────
                sub_cov = cov[np.ix_([j, i], [j, i])]
                vals, vecs = np.linalg.eigh(sub_cov)
                order = vals.argsort()[::-1]
                vals = vals[order]
                vecs = vecs[:, order]
                angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))

                for k, (level, color, label) in enumerate(zip(levels, colors, labels)):
                    width = 2 * np.sqrt(vals[0] * level)
                    height = 2 * np.sqrt(vals[1] * level)
                    ell = Ellipse(
                        xy=(theta_hat[j], theta_hat[i]),
                        width=width,
                        height=height,
                        angle=angle,
                        fill=(k == 0),
                        facecolor=color if k == 0 else "none",
                        alpha=0.20 if k == 0 else 1.0,
                        edgecolor=color,
                        linewidth=1.5,
                        label=label if (j == 0 and i == 1) else None,
                    )
                    ax.add_patch(ell)

                ax.scatter(theta_hat[j], theta_hat[i], s=25, color="k", zorder=5)
                ax.autoscale_view()

                # agrandir légèrement les axes pour que les ellipses soient visibles
                margin_x = 1.5 * np.sqrt(cov[j, j])
                margin_y = 1.5 * np.sqrt(cov[i, i])
                ax.set_xlim(theta_hat[j] - margin_x * 3, theta_hat[j] + margin_x * 3)
                ax.set_ylim(theta_hat[i] - margin_y * 3, theta_hat[i] + margin_y * 3)

            else:
                # ── Triangle supérieur : vide ──────────────────────────────
                ax.axis("off")

            # ── Étiquettes des axes ───────────────────────────────────────
            if i == n - 1:
                ax.set_xlabel(param_names[j], fontsize=11)
            if j == 0 and i != j:
                ax.set_ylabel(param_names[i], fontsize=11)

            ax.tick_params(labelsize=8)

    # Légende globale sur la sous-figure (0,1)
    axes[1, 0].legend(fontsize=8, loc="upper right")

    if title:
        fig.suptitle(title, fontsize=12, y=1.01)

    plt.tight_layout()
    return fig


print("plot_fisher_ellipses() définie.")

### 4.1 Ellipses pour les 4 paramètres SALT3 : `t0, x0, x1, c`

In [ ]:
# ── Extraction depuis le résultat du fit (build_sncosmo_table path) ────────
# `fit_fixed_z` est le résultat de sncosmo.fit_lc (Result object)
# `res`         est le résultat de fit_salt_fink  (dict)

if fit_fixed_z is not None and fit_fixed_z.covariance is not None:
    # Paramètres libres du fit : t0, x0, x1, c
    vpnames = list(fit_fixed_z.vparam_names)  # ['t0','x0','x1','c']
    theta = np.array([fit_fixed_z.parameters[fit_fixed_z.param_names.index(p)] for p in vpnames])
    cov_salt = np.array(fit_fixed_z.covariance)  # (4,4)

    print("Paramètres best-fit :")
    for name, val in zip(vpnames, theta):
        print(f"  {name:>4s} = {val:.6g}")
    print()
    print("Matrice de covariance :")
    print(cov_salt)
else:
    print("fit_fixed_z non disponible ou covariance None — relancer la cellule de fit.")
    cov_salt = None

In [ ]:
# ── Figure 4a : les 4 paramètres SALT3 ──────────────────────────────────────
if cov_salt is not None:
    fig_ell4 = plot_fisher_ellipses(
        param_names=vpnames,
        theta_hat=theta,
        cov=cov_salt,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        title=(f"SALT3 — FinkID {FINK_ID}  z={z_used:.4f}\nEllipses de corrélation Fisher (68 % et 95 % CL)"),
    )
    fig_ell4.savefig(FIGS_DIR / "salt3_fisher_ellipses_4params.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure sauvée → {FIGS_DIR}/salt3_fisher_ellipses_4params.png")
else:
    print("Covariance non disponible.")

### 4.2 Ellipses pour les paramètres physiques seulement : `x1, c`

Ces deux paramètres sont les plus importants pour la cosmologie SNIa :
- `x1` : *stretch* (largeur de la courbe de lumière)
- `c`  : couleur (rougissement)

In [ ]:
# ── Sous-matrice (x1, c) ─────────────────────────────────────────────────────
if cov_salt is not None:
    idx_x1 = vpnames.index("x1")
    idx_c = vpnames.index("c")
    idxs = [idx_x1, idx_c]

    theta_x1c = theta[idxs]
    cov_x1c = cov_salt[np.ix_(idxs, idxs)]

    # corrélation de Pearson x1-c
    rho = cov_x1c[0, 1] / np.sqrt(cov_x1c[0, 0] * cov_x1c[1, 1])
    print(f"x1 = {theta_x1c[0]:.4f} ± {np.sqrt(cov_x1c[0, 0]):.4f}")
    print(f"c  = {theta_x1c[1]:.4f} ± {np.sqrt(cov_x1c[1, 1]):.4f}")
    print(f"Corrélation Pearson ρ(x1, c) = {rho:.4f}")

    fig_x1c = plot_fisher_ellipses(
        param_names=["x1 (stretch)", "c (colour)"],
        theta_hat=theta_x1c,
        cov=cov_x1c,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        figsize=(5, 5),
        title=(f"SALT3 — FinkID {FINK_ID}  z={z_used:.4f}\nEllipses Fisher x1 vs c   ρ = {rho:.3f}"),
    )
    fig_x1c.savefig(FIGS_DIR / "salt3_fisher_ellipses_x1c.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure sauvée → {FIGS_DIR}/salt3_fisher_ellipses_x1c.png")
else:
    print("Covariance non disponible.")